### Normalisation des CSV extraits (gold standard et annotations Medkit).

Ce script applique une normalisation textuelle harmonisée sur les deux fichiers
CSV (gold_standard.csv et annot_medkit.csv) afin de réduire les variations de
forme avant la comparaison et le calcul des métriques.

Transformations appliquées :
- Suppression des accents
- Mise en minuscule
- Suppression de la ponctuation finale
- Séparation chiffres / unités (ex: 40mg → 40 mg)
- Nettoyage des espaces et retours à la ligne
- Mapping des abréviations de voie d'administration (ex: iv → IV, per → per os)
- Déduplication des lignes après normalisation

In [1]:
import pandas as pd
import unicodedata
import re

# Fichiers d'entrée / sortie
GOLD_IN  = "gold_standard.csv"
AUTO_IN  = "annot_medkit.csv"
GOLD_OUT = "gold_standard_norm.csv"
AUTO_OUT = "annot_medkit_norm.csv"

# Colonnes d'attributs à normaliser
ATTRIBUT_COLS = [
    "Dosage", "Frequence", "Voie_administration",
    "Date", "Date_relative", "Contexte"
]



# Dictionnaire de mapping des voies d'administration
VOIE_MAP = {
    "per": "per os", "po": "per os", "p.o.": "per os", "p.o": "per os",
    "iv": "IV", "i.v.": "IV", "i.v": "IV",
    "sc": "SC", "s.c.": "SC", "s.c": "SC",
    "perfusion iv": "perfusion IV",
}



# Fonctions de normalisation
def remove_accents(text: str) -> str:
    "Supprime les accents d'une chaîne de caractères."
    text = unicodedata.normalize("NFD", text)
    return "".join(c for c in text if unicodedata.category(c) != "Mn")


def normalize_text(text: str) -> str:
    """
    Applique une normalisation générique sur un texte :
    minuscule, suppression des accents, nettoyage des espaces,
    séparation chiffres/unités, suppression de la ponctuation finale.
    """
    if pd.isna(text) or str(text).strip() == "":
        return ""
    t = str(text).strip().lower()
    t = re.sub(r"[\r\n\t]+", " ", t)
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"[.\s,;]+$", "", t)
    t = re.sub(r"(\d)([a-zA-Zµ])", r"\1 \2", t)  # 40mg → 40 mg
    t = re.sub(r"([a-zA-Zµ])(\d)", r"\1 \2", t)  # mg40 → mg 40 (cas rare)
    t = remove_accents(t)
    return t.strip()


def normalize_attr(text: str, is_voie: bool = False) -> str:
    """
    Normalise un attribut clinique.
    Si is_voie=True, applique en plus le mapping des abréviations.
    """
    t = normalize_text(text)
    if is_voie:
        t = VOIE_MAP.get(t, t)
    return t


def normalize_med(text: str) -> str:
    """
    Normalise le nom d'un médicament :
    remplace les tirets et underscores par des espaces.
    """
    t = normalize_text(text)
    t = re.sub(r"[-_]+", " ", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip()


# Chargement des CSV bruts
gold = pd.read_csv(GOLD_IN, encoding="utf-8-sig")
auto = pd.read_csv(AUTO_IN, encoding="utf-8-sig")

print(f"Gold chargé : {len(gold)} lignes")
print(f"Auto chargé : {len(auto)} lignes")

Gold chargé : 129 lignes
Auto chargé : 57 lignes


In [2]:
# Application de la normalisation
gold["Medicament_norm"] = gold["Medicament"].apply(normalize_med)
auto["Medicament_norm"] = auto["Medicament"].apply(normalize_med)

for col in ATTRIBUT_COLS:
    is_voie = (col == "Voie_administration")
    gold[col] = gold[col].apply(lambda x: normalize_attr(x, is_voie=is_voie))
    auto[col] = auto[col].apply(lambda x: normalize_attr(x, is_voie=is_voie))


# Déduplication des lignes après normalisation
gold_before = len(gold)
auto_before = len(auto)

dedup_cols = ["Medicament_norm", "Position_texte"] + ATTRIBUT_COLS

gold = gold.drop_duplicates(subset=dedup_cols).reset_index(drop=True)
auto = auto.drop_duplicates(subset=dedup_cols).reset_index(drop=True)

gold["ID"] = range(len(gold))
auto["ID"] = range(len(auto))

print(f"Gold après déduplication : {len(gold)} lignes (supprimés : {gold_before - len(gold)})")
print(f"Auto après déduplication : {len(auto)} lignes (supprimés : {auto_before - len(auto)})")

# Export des CSV normalisés
gold.to_csv(GOLD_OUT, index=False, encoding="utf-8-sig")
auto.to_csv(AUTO_OUT, index=False, encoding="utf-8-sig")

print(f"Fichier exporté : {GOLD_OUT} ({len(gold)} lignes)")
print(f"Fichier exporté : {AUTO_OUT} ({len(auto)} lignes)")

Gold après déduplication : 129 lignes (supprimés : 0)
Auto après déduplication : 57 lignes (supprimés : 0)
Fichier exporté : gold_standard_norm.csv (129 lignes)
Fichier exporté : annot_medkit_norm.csv (57 lignes)
